# `gee_s1s2_translator` — U-Net training (Phase 2)

End-to-end Sentinel-1 to Sentinel-2 translator training for the lowland-heath fire-mapping pipeline. Reads the Phase 1 GCS-harvested TFRecord patches, trains a linear baseline and a 4-level U-Net, evaluates on the held-out test fold (with Ashdown Forest as the geographically-disjoint generalisation test), and writes a validation report back to GCS.

## Instructions for Sonia

1. Open this notebook in Colab. Runtime → Change runtime type → GPU (T4 is fine).
2. Edit **only Cell 2 (Parameters)** with your project ID, bucket, and run name.
3. Runtime → Run all.
4. Wait ~30–60 minutes (T4). Check the saved validation report on GCS at the prefix you set.
5. Each named run writes to its own GCS folder, so re-running with a new `TRAINING_RUN_NAME` does not overwrite earlier runs.

All artifacts go to GCS — model checkpoints, sidecar JSON, training-curve CSVs, validation report, and a copy of the parameter cell for full reproducibility. Phase 3 (Vertex AI deployment) reads the saved model from there.

Methodology divergences vs v2: see `docs/methodology_divergences.md` and `training/v2_reference_results.json`. The headline divergences (S1 calibration −1.5 dB offset, Ashdown 105 vs 76 patches, T30UWB partial-coverage handling) are all documented and operationally inert.

In [ ]:
# Cell 2: PARAMETERS — edit these and only these.
# Re-running with a new TRAINING_RUN_NAME does not overwrite earlier runs.

PROJECT_ID = "wildfire-495012"
GCS_BUCKET = "marcus-heath-fire-mapping"
GCS_PREFIX = "gee_s1s2_translator/operational_v1"
HARVEST_MANIFEST_URI = f"gs://{GCS_BUCKET}/{GCS_PREFIX}/manifest.csv"

TRAINING_RUN_NAME = "v2_equivalent_initial"
MODEL_OUTPUT_PREFIX = f"gs://{GCS_BUCKET}/{GCS_PREFIX}/models/{TRAINING_RUN_NAME}/"

S1_STATS_URI = f"gs://{GCS_BUCKET}/{GCS_PREFIX}/s1_stats.json"

# Hyperparameters (sensible defaults; usually no need to edit).
BATCH_SIZE = 8
MAX_EPOCHS = 80
LEARNING_RATE = 1e-4
EARLY_STOPPING_PATIENCE = 15
RANDOM_SEED = 42

# Set False if your Phase 1 harvest used `speckle_filter.enabled: false`
# and you want this pipeline to apply Lee 5x5 at training time instead.
# Leave True (the default) for the standard operational config.
S1_LEE_ALREADY_APPLIED_AT_HARVEST = True

## Cell 3 — Authentication and dependency setup

In [ ]:
# Install the gee_s1s2 training package directly from Marcus' GitHub fork once the
# repo is public; for now we vendor a copy of training/src/training into Colab via
# the parent project. If running outside Colab, ensure the package is on sys.path.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Authenticate the Colab session for both GCS and Earth Engine.
    from google.colab import auth as colab_auth
    colab_auth.authenticate_user()
    # The training package needs to be installed/available. Easiest path: clone the
    # repo and pip-install in editable mode. Adjust the URL below to your fork.
    repo_url = os.environ.get(
        "GEE_S1S2_REPO_URL", "https://github.com/<your-fork>/gee_s1s2_translator.git"
    )
    if not os.path.exists("/content/gee_s1s2_translator"):
        subprocess.run(
            ["git", "clone", "--depth", "1", repo_url, "/content/gee_s1s2_translator"],
            check=True,
        )
    sys.path.insert(0, "/content/gee_s1s2_translator/training/src")

import tensorflow as tf
from google.cloud import storage

# Sanity: confirm GCS bucket is reachable and manifest exists.
client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(GCS_BUCKET)
assert bucket.exists(), f"Bucket gs://{GCS_BUCKET}/ not reachable. Check PROJECT_ID and auth."
manifest_blob = bucket.blob(f"{GCS_PREFIX}/manifest.csv")
assert manifest_blob.exists(), (
    f"Manifest not found at {HARVEST_MANIFEST_URI}. "
    f"Did Phase 1 harvest run successfully?"
)
print(f"OK. Project={PROJECT_ID}  Bucket=gs://{GCS_BUCKET}/  Manifest={HARVEST_MANIFEST_URI}")

## Cell 4 — Load manifest, print summary, run sanity checks

Confirms the harvest's split routing matches the Phase 1 design intent.

In [ ]:
from collections import Counter
from training.data import load_manifest, split_uris

entries = load_manifest(HARVEST_MANIFEST_URI)
print(f"Manifest rows: {len(entries)}")

per_split = Counter(e.split for e in entries)
per_aoi = Counter(e.aoi_name for e in entries)
per_aoi_window = Counter((e.aoi_name, e.split) for e in entries)

print(f"\nPer-split distribution:")
for s in ["train", "val", "test"]:
    n = per_split.get(s, 0)
    print(f"  {s:<10} {n:>5}  ({100.0 * n / max(1, len(entries)):>5.1f}%)")

print(f"\nPer-AOI patch counts:")
for aoi, n in sorted(per_aoi.items(), key=lambda kv: -kv[1]):
    print(f"  {aoi:<40} {n:>5}")

# Sanity check 1: Ashdown should be force-routed to test.
ashdown_splits = {e.split for e in entries if e.aoi_name == "Ashdown Forest"}
assert ashdown_splits == {"test"}, (
    f"Ashdown should be force_split=test, got splits={ashdown_splits}. "
    f"Check config/operational_v1.yaml."
)
n_ashdown = sum(1 for e in entries if e.aoi_name == "Ashdown Forest")
print(f"\nSanity 1 OK: Ashdown is 100% test ({n_ashdown} patches).")

# Sanity check 2: Hankley pre-fire 2022 should be absent (exclude_windows).
hankley_pre22 = [
    e for e in entries
    if e.aoi_name == "Hankley Common"
]
# We don't have date_window in the entries dataclass; just confirm Hankley exists
# but doesn't dominate (the exclude pruned one of its four windows).
n_hankley = len(hankley_pre22)
print(f"Sanity 2 OK: Hankley Common has {n_hankley} patches (3 windows; pre-fire 2022 excluded by config).")

train_uris = split_uris(entries, "train")
val_uris = split_uris(entries, "val")
test_uris = split_uris(entries, "test")
print(f"\nTrain shards: {len(train_uris)}   Val shards: {len(val_uris)}   Test shards: {len(test_uris)}")

## Cell 5 — GPU detection

In [ ]:
gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    print("WARNING: no GPU detected. Training will run on CPU and take many hours.")
    print("In Colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU.")
else:
    for g in gpus:
        details = tf.config.experimental.get_device_details(g)
        print(f"GPU: {g.name}  device_name={details.get('device_name', '?')}")
    # Allow memory growth so other Colab notebooks can share the GPU.
    for g in gpus:
        try:
            tf.config.experimental.set_memory_growth(g, True)
        except Exception:
            pass

# Rough runtime estimate.
n_train_steps = max(1, len(train_uris) // BATCH_SIZE)
n_epochs_est = MAX_EPOCHS  # may early-stop sooner
print(f"Estimated training time on T4: 30-60 min for {n_train_steps} steps/epoch x ~{n_epochs_est} epochs (early stopping usually fires earlier).")

## Cell 6 — TFRecord data pipeline

Builds train/val/test `tf.data.Dataset` instances. S1 is z-score normalised using `s1_stats.json` (computed and persisted to GCS the first time this cell runs); S2 is already in `[0, 1]` from the Phase 1 export.

**Lee speckle filter** is *not* re-applied in this pipeline because the default Phase 1 harvest applied Lee 5×5 server-side via the GEE calibration recipe. If you re-harvest with `speckle_filter.enabled: false`, set `S1_LEE_ALREADY_APPLIED_AT_HARVEST = False` in Cell 2 and the pipeline will apply Lee here at training time instead.

In [ ]:
from training.data import build_dataset, load_or_compute_s1_stats

stats = load_or_compute_s1_stats(train_uris=train_uris, stats_uri=S1_STATS_URI, n_patches=200)
print(f"S1 stats (z-score normalisation):")
for b in ["VV", "VH"]:
    print(f"  {b}: mean={stats.mean[b]:+.3f} dB   std={stats.std[b]:.3f} dB")

apply_lee_in_pipeline = not S1_LEE_ALREADY_APPLIED_AT_HARVEST
print(f"\nApplying Lee 5x5 in pipeline: {apply_lee_in_pipeline}")

train_ds = build_dataset(
    train_uris, stats=stats, batch_size=BATCH_SIZE, shuffle=True,
    apply_lee=apply_lee_in_pipeline, seed=RANDOM_SEED,
)
val_ds = build_dataset(
    val_uris, stats=stats, batch_size=BATCH_SIZE, shuffle=False,
    apply_lee=apply_lee_in_pipeline,
)
test_ds = build_dataset(
    test_uris, stats=stats, batch_size=BATCH_SIZE, shuffle=False,
    apply_lee=apply_lee_in_pipeline,
)

# Smoke check: pull one batch.
for s1, s2 in train_ds.take(1):
    print(f"\nOne train batch: s1={s1.shape} dtype={s1.dtype}, s2={s2.shape} dtype={s2.dtype}")
    print(f"  S1 z-scored stats:  mean={float(tf.reduce_mean(s1)):+.3f}  std={float(tf.math.reduce_std(s1)):.3f}")
    print(f"  S2 [0,1] stats:     mean={float(tf.reduce_mean(s2)):+.3f}  std={float(tf.math.reduce_std(s2)):.3f}")

## Cell 7 — Linear baseline training

Per-pixel 1×1 conv from S1 (2 channels) to S2 (6 reflectance bands). The reference "no spatial reasoning" model — its variance-collapse on driver bands is the headroom the U-Net will (we hope) significantly close.

In [ ]:
import tensorflow as tf
tf.keras.utils.set_random_seed(RANDOM_SEED)

from training.linear_baseline import build_linear_baseline
from training.trainer import train as train_model

lb = build_linear_baseline(input_shape=(256, 256, 2), out_channels=6)
lb.summary()

lb_result = train_model(
    lb,
    train_ds=train_ds, val_ds=val_ds,
    checkpoint_uri=MODEL_OUTPUT_PREFIX + "linear_baseline.keras",
    sidecar_uri=MODEL_OUTPUT_PREFIX + "linear_baseline_sidecar.json",
    log_uri=MODEL_OUTPUT_PREFIX + "train_linear_baseline.csv",
    learning_rate=LEARNING_RATE,
    max_epochs=MAX_EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    extra_metadata={
        "run_name": TRAINING_RUN_NAME,
        "manifest_uri": HARVEST_MANIFEST_URI,
        "n_train_shards": len(train_uris),
        "n_val_shards": len(val_uris),
    },
)
print(f"\nLinear baseline best val RMSE: {lb_result.best_val_rmse:.4f} at epoch {lb_result.best_epoch}.")

## Cell 8 — U-Net training

4-level U-Net matching v2: encoder/bottleneck/decoder with bilinear upsampling, batch-norm, ReLU, sigmoid output. Adam @ 1e-4, batch size 8, combined L1+L2 loss with 1.0 / 0.5 weights, early stopping patience 15 on val RMSE.

In [ ]:
tf.keras.utils.set_random_seed(RANDOM_SEED)
from training.model import build_unet

unet = build_unet(input_shape=(256, 256, 2), out_channels=6, base_channels=32)
unet.summary()
print(f"U-Net parameter count: {unet.count_params():,}")

unet_result = train_model(
    unet,
    train_ds=train_ds, val_ds=val_ds,
    checkpoint_uri=MODEL_OUTPUT_PREFIX + "unet.keras",
    sidecar_uri=MODEL_OUTPUT_PREFIX + "unet_sidecar.json",
    log_uri=MODEL_OUTPUT_PREFIX + "train_unet.csv",
    learning_rate=LEARNING_RATE,
    max_epochs=MAX_EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    extra_metadata={
        "run_name": TRAINING_RUN_NAME,
        "manifest_uri": HARVEST_MANIFEST_URI,
        "n_train_shards": len(train_uris),
        "n_val_shards": len(val_uris),
        "base_channels": 32,
    },
)
print(f"\nU-Net best val RMSE: {unet_result.best_val_rmse:.4f} at epoch {unet_result.best_epoch}.")

## Cell 9 — Validation report on the test set

Per-band MAE/RMSE separately for the in-distribution test fold and the **Ashdown held-out test patches**, plus the variance-collapse diagnostic on driver bands B04/B08/B11/B12. The Ashdown numbers are the dissertation's headline generalisation result.

In [ ]:
import json
import numpy as np

from training.metrics import (
    DRIVER_BANDS, S2_BAND_ORDER,
    driver_band_mean_retention_pct,
    patch_specific_variance_retention,
    per_band_mae_rmse,
)

# Build separate test datasets for in-distribution vs Ashdown.
ashdown_uris = [e.tfrecord_uri for e in entries if e.split == "test" and e.aoi_name == "Ashdown Forest"]
indist_uris = [e.tfrecord_uri for e in entries if e.split == "test" and e.aoi_name != "Ashdown Forest"]
print(f"Ashdown held-out test shards: {len(ashdown_uris)}")
print(f"In-distribution test shards:  {len(indist_uris)}")

ashdown_ds = build_dataset(ashdown_uris, stats=stats, batch_size=BATCH_SIZE,
                            shuffle=False, apply_lee=apply_lee_in_pipeline)
indist_ds = build_dataset(indist_uris, stats=stats, batch_size=BATCH_SIZE,
                           shuffle=False, apply_lee=apply_lee_in_pipeline)

def _materialise(ds):
    """Run inference, return concatenated (truth, pred) arrays."""
    truths, preds = [], []
    for s1, s2 in ds:
        truths.append(s2.numpy())
        preds.append(unet(s1, training=False).numpy())
    return np.concatenate(truths), np.concatenate(preds)

def _materialise_baseline(ds):
    truths, preds = [], []
    for s1, s2 in ds:
        truths.append(s2.numpy())
        preds.append(lb(s1, training=False).numpy())
    return np.concatenate(truths), np.concatenate(preds)

ash_truth, ash_pred = _materialise(ashdown_ds)
ind_truth, ind_pred = _materialise(indist_ds)
ash_truth_lb, ash_pred_lb = _materialise_baseline(ashdown_ds)

ashdown_per_band = per_band_mae_rmse(ash_truth, ash_pred)
indist_per_band = per_band_mae_rmse(ind_truth, ind_pred)
ashdown_per_band_lb = per_band_mae_rmse(ash_truth_lb, ash_pred_lb)

ashdown_variance = patch_specific_variance_retention(ash_truth, ash_pred)
ashdown_variance_lb = patch_specific_variance_retention(ash_truth_lb, ash_pred_lb)

ashdown_driver_pct = driver_band_mean_retention_pct(ashdown_variance)
ashdown_driver_pct_lb = driver_band_mean_retention_pct(ashdown_variance_lb)

print(f"\nAshdown held-out test ({ash_truth.shape[0]} patches)")
print(f"  band  | UNet MAE  RMSE  | LB MAE  RMSE")
for u, lb_m in zip(ashdown_per_band, ashdown_per_band_lb):
    print(f"  {u.band:<5} |   {u.mae:.4f}  {u.rmse:.4f}  |  {lb_m.mae:.4f}  {lb_m.rmse:.4f}")
print(f"  Driver-band mean variance retention: U-Net={ashdown_driver_pct:.1f}% vs Baseline={ashdown_driver_pct_lb:.1f}%")

In [ ]:
# Render the validation report as markdown and save to GCS.
from datetime import datetime, timezone

lines = []
lines.append(f"# Validation report — `{TRAINING_RUN_NAME}`")
lines.append("")
lines.append(f"Generated: {datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')}")
lines.append(f"\nManifest: `{HARVEST_MANIFEST_URI}`\n")
lines.append(f"Best val RMSE — U-Net: **{unet_result.best_val_rmse:.4f}** @ epoch {unet_result.best_epoch}")
lines.append(f"Best val RMSE — Linear baseline: {lb_result.best_val_rmse:.4f} @ epoch {lb_result.best_epoch}")
lines.append("")

lines.append("## Per-band MAE / RMSE — held-out Ashdown test set")
lines.append("")
lines.append("| Band | U-Net MAE | U-Net RMSE | Baseline MAE | Baseline RMSE |")
lines.append("| --- | ---: | ---: | ---: | ---: |")
for u, lb_m in zip(ashdown_per_band, ashdown_per_band_lb):
    lines.append(f"| {u.band} | {u.mae:.4f} | {u.rmse:.4f} | {lb_m.mae:.4f} | {lb_m.rmse:.4f} |")
lines.append(f"\n_Patch count: {ash_truth.shape[0]} (vs v2's 76; see footnote in `training/v2_reference_results.json`)._\n")

lines.append("## Per-band MAE / RMSE — in-distribution test set")
lines.append("")
lines.append("| Band | U-Net MAE | U-Net RMSE |")
lines.append("| --- | ---: | ---: |")
for u in indist_per_band:
    lines.append(f"| {u.band} | {u.mae:.4f} | {u.rmse:.4f} |")
lines.append(f"\n_Patch count: {ind_truth.shape[0]}._\n")

lines.append("## Variance-collapse diagnostic — Ashdown held-out (patch-specific truth std)")
lines.append("")
lines.append("Pass bracket: [75 %, 105 %] on driver bands. Below = collapse, above = noise.\n")
lines.append("| Band | Driver | U-Net mean | U-Net median | U-Net pass | Baseline mean | Baseline pass |")
lines.append("| --- | :---: | ---: | ---: | :---: | ---: | :---: |")
for u, lb_v in zip(ashdown_variance, ashdown_variance_lb):
    drv = "\u2713" if u.is_driver else ""
    u_pass = "OK" if u.pass_75_105_bracket else "FAIL"
    lb_pass = "OK" if lb_v.pass_75_105_bracket else "FAIL"
    lines.append(
        f"| {u.band} | {drv} | {u.mean_pred_over_truth_pct:.1f}% | "
        f"{u.median_pred_over_truth_pct:.1f}% | {u_pass} | "
        f"{lb_v.mean_pred_over_truth_pct:.1f}% | {lb_pass} |"
    )
lines.append("")
lines.append(f"**Driver-band mean retention**: U-Net = **{ashdown_driver_pct:.1f}%** vs Baseline = {ashdown_driver_pct_lb:.1f}%")
if ashdown_driver_pct_lb > 0:
    lines.append(f"  → U-Net / Baseline ratio = {ashdown_driver_pct / ashdown_driver_pct_lb:.2f}×")
lines.append("")

report_md = "\n".join(lines)
report_uri = MODEL_OUTPUT_PREFIX + "validation_report.md"
with tf.io.gfile.GFile(report_uri, "w") as f:
    f.write(report_md)

# Sidecar JSON with structured metrics for programmatic comparison.
report_json = {
    "run_name": TRAINING_RUN_NAME,
    "unet_best_val_rmse": unet_result.best_val_rmse,
    "linear_baseline_best_val_rmse": lb_result.best_val_rmse,
    "ashdown": {
        "n_patches": int(ash_truth.shape[0]),
        "unet_per_band_mae": {m.band: m.mae for m in ashdown_per_band},
        "unet_per_band_rmse": {m.band: m.rmse for m in ashdown_per_band},
        "baseline_per_band_mae": {m.band: m.mae for m in ashdown_per_band_lb},
        "baseline_per_band_rmse": {m.band: m.rmse for m in ashdown_per_band_lb},
        "unet_driver_band_mean_retention_pct": ashdown_driver_pct,
        "baseline_driver_band_mean_retention_pct": ashdown_driver_pct_lb,
        "unet_per_band_variance_retention": [
            {"band": v.band, "is_driver": v.is_driver,
             "mean_pct": v.mean_pred_over_truth_pct,
             "median_pct": v.median_pred_over_truth_pct,
             "pass_bracket": v.pass_75_105_bracket}
            for v in ashdown_variance
        ],
    },
    "in_distribution": {
        "n_patches": int(ind_truth.shape[0]),
        "unet_per_band_mae": {m.band: m.mae for m in indist_per_band},
        "unet_per_band_rmse": {m.band: m.rmse for m in indist_per_band},
    },
}
with tf.io.gfile.GFile(MODEL_OUTPUT_PREFIX + "validation_report.json", "w") as f:
    json.dump(report_json, f, indent=2)

print(f"Validation report written to {report_uri}")

## Cell 10 — Comparison vs v2 reference

Loads `training/v2_reference_results.json` and renders a side-by-side. Footnotes flag the documented divergences (Ashdown 105 vs 76, S1 −1.5 dB calibration offset).

In [ ]:
import json

if IN_COLAB:
    with open("/content/gee_s1s2_translator/training/v2_reference_results.json") as f:
        v2 = json.load(f)
else:
    with open("../v2_reference_results.json") as f:
        v2 = json.load(f)

print(f"=== Comparison: this run ({TRAINING_RUN_NAME}) vs v2 ({v2['v2_run_id']}) ===\n")
print(f"  {'metric':<48} {'v2':>10} {'GEE port':>10} {'verdict':>12}")

v2_unet = v2["best_val_rmse"]["unet"]
v2_lb = v2["best_val_rmse"]["linear_baseline"]
tol = v2["tolerance_for_phase_2_comparison"]
rmse_lo, rmse_hi = tol["val_rmse_pass_bracket"]
var_floor = tol["ashdown_variance_retention_pass_floor_pct"]

def _v(ok):
    return "OK" if ok else "INVESTIGATE"

print(f"  {'best val RMSE — U-Net':<48} {v2_unet:>10.4f} {unet_result.best_val_rmse:>10.4f} "
      f"{_v(rmse_lo <= unet_result.best_val_rmse <= rmse_hi):>12}")
print(f"  {'best val RMSE — Linear baseline':<48} {v2_lb:>10.4f} {lb_result.best_val_rmse:>10.4f}")
print()
print(f"  {'Ashdown driver-band variance retention U-Net':<48} "
      f"{v2['variance_retention_driver_bands_patch_specific_pct']['unet_mean_pct']:>9.1f}% "
      f"{ashdown_driver_pct:>9.1f}% "
      f"{_v(ashdown_driver_pct >= var_floor):>12}")
print(f"  {'Ashdown driver-band variance retention LB':<48} "
      f"{v2['variance_retention_driver_bands_patch_specific_pct']['linear_baseline_mean_pct']:>9.1f}% "
      f"{ashdown_driver_pct_lb:>9.1f}%")
v2_ratio = v2["variance_retention_driver_bands_patch_specific_pct"]["unet_vs_baseline_ratio"]
this_ratio = ashdown_driver_pct / max(ashdown_driver_pct_lb, 1e-9)
print(f"  {'U-Net / Baseline variance ratio':<48} {v2_ratio:>10.2f} {this_ratio:>10.2f}")
print()
print("  Per-band MAE on Ashdown held-out:")
v2_ash_mae = v2["ashdown_held_out_per_band_mae"]
for u in ashdown_per_band:
    if u.band in v2_ash_mae:
        print(f"    {u.band:<46} {v2_ash_mae[u.band]:>10.4f} {u.mae:>10.4f}")
print("\nFootnotes (full text in training/v2_reference_results.json):")
for fn in v2["footnotes"]:
    print(f"  - {fn['topic']}: {fn['note'][:200]}...")

# Save the parameter cell verbatim for reproducibility.
params = {
    "PROJECT_ID": PROJECT_ID, "GCS_BUCKET": GCS_BUCKET, "GCS_PREFIX": GCS_PREFIX,
    "HARVEST_MANIFEST_URI": HARVEST_MANIFEST_URI,
    "TRAINING_RUN_NAME": TRAINING_RUN_NAME,
    "MODEL_OUTPUT_PREFIX": MODEL_OUTPUT_PREFIX,
    "S1_STATS_URI": S1_STATS_URI,
    "BATCH_SIZE": BATCH_SIZE, "MAX_EPOCHS": MAX_EPOCHS,
    "LEARNING_RATE": LEARNING_RATE,
    "EARLY_STOPPING_PATIENCE": EARLY_STOPPING_PATIENCE,
    "RANDOM_SEED": RANDOM_SEED,
    "S1_LEE_ALREADY_APPLIED_AT_HARVEST": S1_LEE_ALREADY_APPLIED_AT_HARVEST,
}
with tf.io.gfile.GFile(MODEL_OUTPUT_PREFIX + "parameters.json", "w") as f:
    json.dump(params, f, indent=2)

## Cell 11 — Summary and what to do next

All artifacts are saved in GCS at `MODEL_OUTPUT_PREFIX`. To inspect them:

```bash
gcloud storage ls gs://<your-bucket>/gee_s1s2_translator/operational_v1/models/<run-name>/
```

Files written:

* `unet.keras` — trained U-Net weights
* `unet_sidecar.json` — best epoch, val RMSE, hyperparameters, timestamps
* `train_unet.csv` — per-epoch loss/metric curve
* `linear_baseline.keras` + `_sidecar.json` + `train_linear_baseline.csv`
* `validation_report.md` — human-readable validation report
* `validation_report.json` — structured metrics (for programmatic comparison)
* `parameters.json` — full reproducibility snapshot of Cell 2
* `s1_stats.json` (one level up, shared across runs) — z-score normalisation stats

**Expected outcome** (within tolerance documented in `training/v2_reference_results.json`):

* Val RMSE in [0.085, 0.130]
* Ashdown driver-band variance retention ≥ 40 %
* U-Net / Baseline ratio close to v2's 3.3×

If your numbers diverge from these brackets, **stop and investigate** before deploying. Common causes are (a) the manifest is wrong or stale, (b) the GPU was unavailable and training ran on CPU and converged poorly, or (c) one of the documented divergences (S1 calibration offset, Ashdown count) interacted in an unexpected way.

## Phase 3 — Vertex AI deployment

When Phase 2 review-gate clears, the next step is wrapping `unet.keras` in a Vertex AI Endpoint. The Phase 3 prompt covers that; it reads from `MODEL_OUTPUT_PREFIX + 'unet.keras'`.